# BTC Hourly Candlestick Features

Extended version of `OHLCV-hourly.ipynb` focused on **candlestick anatomy and pattern detection**.

**Categories**: Candlestick Anatomy · Candlestick Pattern Detection

**Note**: Dec 2017 is downloaded as warmup to allow pattern detection rolling windows (e.g., Morning/Evening Star median body lookback of 50 candles) to be fully warmed up by 2018-01-01.

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

# --- 1. CONFIGURATION ---
symbol = 'BTCUSDT'
interval = '1h'
base_url = "https://data.binance.vision/data/spot/monthly/klines"
all_frames = []
EPS = 1e-10  # Epsilon to prevent division-by-zero

print(f"\U0001f4e1 Downloading {symbol} Hourly Data (Dec 2017 - Dec 2025)...")

# --- 2. DOWNLOAD LOOP ---
# Dec 2017 is fetched to "warm up" rolling windows.
# Pattern detection (Morning/Evening Star) uses a 50-candle rolling median for
# body significance thresholding. Dec 2017 provides ~744 hourly candles,
# so all patterns are fully valid well before Jan 1, 2018.
fetch_list = [(2017, 12)] + [(y, m) for y in range(2018, 2026) for m in range(1, 13)]

for year, month in fetch_list:
    m_str = f"{month:02d}"
    file_name = f"{symbol}-{interval}-{year}-{m_str}.zip"
    url = f"{base_url}/{symbol}/{interval}/{file_name}"
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                csv_name = file_name.replace('.zip', '.csv')
                with z.open(csv_name) as f:
                    all_frames.append(pd.read_csv(f, header=None))
            print(f"\u2705 Loaded {year}-{m_str}")
        else:
            print(f"\u26a0\ufe0f Skipped {year}-{m_str}: Status {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"\u274c Failed to download {year}-{m_str}: {e}")

📡 Downloading BTCUSDT Hourly Data (Dec 2017 - Dec 2025)...
✅ Loaded 2017-12
✅ Loaded 2018-01
✅ Loaded 2018-02
✅ Loaded 2018-03
✅ Loaded 2018-04
✅ Loaded 2018-05
✅ Loaded 2018-06
✅ Loaded 2018-07
✅ Loaded 2018-08
✅ Loaded 2018-09
✅ Loaded 2018-10
✅ Loaded 2018-11
✅ Loaded 2018-12
✅ Loaded 2019-01
✅ Loaded 2019-02
✅ Loaded 2019-03
✅ Loaded 2019-04
✅ Loaded 2019-05
✅ Loaded 2019-06
✅ Loaded 2019-07
✅ Loaded 2019-08
✅ Loaded 2019-09
✅ Loaded 2019-10
✅ Loaded 2019-11
✅ Loaded 2019-12
✅ Loaded 2020-01
✅ Loaded 2020-02
✅ Loaded 2020-03
✅ Loaded 2020-04
✅ Loaded 2020-05
✅ Loaded 2020-06
✅ Loaded 2020-07
✅ Loaded 2020-08
✅ Loaded 2020-09
✅ Loaded 2020-10
✅ Loaded 2020-11
✅ Loaded 2020-12
✅ Loaded 2021-01
✅ Loaded 2021-02
✅ Loaded 2021-03
✅ Loaded 2021-04
✅ Loaded 2021-05
✅ Loaded 2021-06
✅ Loaded 2021-07
✅ Loaded 2021-08
✅ Loaded 2021-09
✅ Loaded 2021-10
✅ Loaded 2021-11
✅ Loaded 2021-12
✅ Loaded 2022-01
✅ Loaded 2022-02
✅ Loaded 2022-03
✅ Loaded 2022-04
✅ Loaded 2022-05
✅ Loaded 2022-06
✅ Load

In [2]:
# --- 3. DYNAMIC TIMESTAMP CONVERSION ---
# Binance switched from millisecond to microsecond timestamps around mid-2024.
# This function handles both formats transparently.

if all_frames:
    df = pd.concat(all_frames, ignore_index=True)
    df.columns = ['open_time', 'open', 'high', 'low', 'close', 'volume',
                  'close_time', 'q_vol', 'count', 'tb_base', 'tb_quote', 'ignore']

    def convert_binance_time(ts):
        """Handle Binance's mid-2024 shift from millisecond to microsecond timestamps."""
        if ts > 10**14:
            return pd.to_datetime(ts, unit='us', utc=True)  # Microseconds
        return pd.to_datetime(ts, unit='ms', utc=True)      # Milliseconds

    print("\u23f3 Converting timestamps to strictly formatted UTC...")
    df['timestamp'] = df['open_time'].apply(convert_binance_time)

    df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume',
             'q_vol', 'count', 'tb_base']].copy()

    print(f"\u2705 Timestamp conversion complete. Total raw rows: {len(df)}")
else:
    raise RuntimeError("No data was downloaded. Check your network connection.")

⏳ Converting timestamps to strictly formatted UTC...
✅ Timestamp conversion complete. Total raw rows: 70752


## Feature Engineering — Candlestick Features

Two sub-categories:
1. **Candlestick Anatomy** — structural decomposition of each bar (body, shadows, ratios)
2. **Candlestick Pattern Detection** — classic Japanese patterns as binary flags

In [3]:
# ═══════════════════════════════════════════════════════════════
# 4.1 CANDLESTICK ANATOMY (7 features)
#
# These decompose each candle into its structural components,
# giving the model explicit numerical representations of what
# the candlestick images encode visually.
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Candlestick Anatomy features...")

# Body: signed (positive = bullish) and absolute magnitude
df['body_size']     = df['close'] - df['open']
df['body_size_abs'] = df['body_size'].abs()

# Shadows: rejection wicks above and below
df['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']

# Total candle range (high−low): a volatility proxy per bar
df['candle_range'] = df['high'] - df['low']

# Body-to-range ratio: how decisive the bar was
#   → Marubozu (full body, no wicks) ≈ 1.0
#   → Doji (tiny body, big wicks)    ≈ 0.0
df['body_to_range_ratio'] = df['body_size_abs'] / (df['candle_range'] + EPS)

# Shadow asymmetry ratio: upper / lower shadow
#   → High ratio = strong rejection from highs (bearish pressure)
#   → Low ratio  = strong rejection from lows (bullish pressure)
df['shadow_ratio'] = df['upper_shadow'] / (df['lower_shadow'] + EPS)

print("   \u2705 Anatomy: body_size, body_size_abs, upper_shadow, lower_shadow, candle_range, body_to_range_ratio, shadow_ratio")

📊 Computing Candlestick Anatomy features...
   ✅ Anatomy: body_size, body_size_abs, upper_shadow, lower_shadow, candle_range, body_to_range_ratio, shadow_ratio


In [4]:
# ═══════════════════════════════════════════════════════════════
# 4.2 CANDLESTICK PATTERN DETECTION (8 binary features)
#
# Classic Japanese candlestick patterns widely cited in
# financial technical analysis. Each is a binary 0/1 flag.
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Detecting Candlestick Patterns...")

body  = df['body_size']
abody = df['body_size_abs']
ushad = df['upper_shadow']
lshad = df['lower_shadow']
ratio = df['body_to_range_ratio']

prev_body  = body.shift(1)
prev_abody = abody.shift(1)
prev_open  = df['open'].shift(1)
prev_close = df['close'].shift(1)

# For 3-bar patterns (Morning/Evening Star)
prev2_body  = body.shift(2)
prev2_abody = abody.shift(2)

# Rolling median body size for significance thresholding
# (a body is "significant" if it exceeds the recent median)
median_body = abody.rolling(window=50, min_periods=20).median()

# ── DOJI ──
# Body is less than 10% of total range → indecision candle
df['is_doji'] = (ratio < 0.1).astype(int)

# ── HAMMER ──
# Long lower shadow (≥ 2x body), small upper shadow
# Bullish reversal signal at bottoms
df['is_hammer'] = (
    (lshad > 2 * abody) & (ushad < abody)
).astype(int)

# ── INVERTED HAMMER ──
# Long upper shadow (≥ 2x body), small lower shadow
# Potential bullish reversal at bottoms
df['is_inverted_hammer'] = (
    (ushad > 2 * abody) & (lshad < abody)
).astype(int)

# ── BULLISH ENGULFING ──
# Previous bearish candle fully engulfed by current bullish candle
df['is_engulfing_bullish'] = (
    (prev_body < 0) &              # Previous was bearish
    (body > 0) &                    # Current is bullish
    (df['open'] <= prev_close) &    # Current open at or below prev close
    (df['close'] >= prev_open)      # Current close at or above prev open
).astype(int)

# ── BEARISH ENGULFING ──
# Previous bullish candle fully engulfed by current bearish candle
df['is_engulfing_bearish'] = (
    (prev_body > 0) &              # Previous was bullish
    (body < 0) &                    # Current is bearish
    (df['open'] >= prev_close) &    # Current open at or above prev close
    (df['close'] <= prev_open)      # Current close at or below prev open
).astype(int)

# ── MORNING STAR (3-bar bullish reversal) ──
# Bar1: large bearish | Bar2: small body (star) | Bar3: large bullish
df['is_morning_star'] = (
    (prev2_body < 0) &                         # Bar 1: bearish
    (prev2_abody > median_body.shift(2)) &      # Bar 1: significant body
    (prev_abody < 0.3 * prev2_abody) &          # Bar 2: small body (star)
    (body > 0) &                                # Bar 3: bullish
    (abody > median_body)                        # Bar 3: significant body
).astype(int)

# ── EVENING STAR (3-bar bearish reversal) ──
# Bar1: large bullish | Bar2: small body (star) | Bar3: large bearish
df['is_evening_star'] = (
    (prev2_body > 0) &                          # Bar 1: bullish
    (prev2_abody > median_body.shift(2)) &       # Bar 1: significant body
    (prev_abody < 0.3 * prev2_abody) &           # Bar 2: small body (star)
    (body < 0) &                                 # Bar 3: bearish
    (abody > median_body)                         # Bar 3: significant body
).astype(int)

# ── MARUBOZU ──
# Body fills >95% of the total range (almost no shadows)
# Strong directional conviction
df['is_marubozu'] = (ratio > 0.95).astype(int)

print("   \u2705 Patterns: is_doji, is_hammer, is_inverted_hammer, is_engulfing_bullish, is_engulfing_bearish, is_morning_star, is_evening_star, is_marubozu")

📊 Detecting Candlestick Patterns...
   ✅ Patterns: is_doji, is_hammer, is_inverted_hammer, is_engulfing_bullish, is_engulfing_bearish, is_morning_star, is_evening_star, is_marubozu


In [5]:
# --- 5. FILTER & EXPORT ---
print("\u2702\ufe0f Filtering data to start exactly from 2018-01-01 00:00:00 UTC...")
df = df.dropna().reset_index(drop=True)
df = df[df['timestamp'] >= '2018-01-01'].reset_index(drop=True)

# Add Trend Label
df['trend'] = np.where(df['close'] > df['open'], 'Bullish', 'Bearish')

# Export
df.to_csv('btc_hourly_candlestick.csv', index=False)

print(f"\u2728 Success! Total Rows: {len(df)}")
print(f"Data starts at: {df['timestamp'].min()}")
print(f"Data ends at: {df['timestamp'].max()}")
print(f"Total features: {len(df.columns)}")
print(f"Features: {list(df.columns)}")

✂️ Filtering data to start exactly from 2018-01-01 00:00:00 UTC...
✨ Success! Total Rows: 70008
Data starts at: 2018-01-01 00:00:00+00:00
Data ends at: 2025-12-31 23:00:00+00:00
Total features: 25
Features: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'q_vol', 'count', 'tb_base', 'body_size', 'body_size_abs', 'upper_shadow', 'lower_shadow', 'candle_range', 'body_to_range_ratio', 'shadow_ratio', 'is_doji', 'is_hammer', 'is_inverted_hammer', 'is_engulfing_bullish', 'is_engulfing_bearish', 'is_morning_star', 'is_evening_star', 'is_marubozu', 'trend']


In [6]:
# --- 6. VERIFICATION ---
print("\n" + "=" * 60)
print("  VERIFICATION REPORT")
print("=" * 60)

# Trend distribution
trend_counts = df['trend'].value_counts()
print(f"\nTrend Distribution:")
print(trend_counts)

print(f"\nNaN Check: {df.isna().sum().sum()} remaining NaNs")

# Pattern hit rates
print(f"\nCandlestick Pattern Hit Rates:")
pattern_cols = [c for c in df.columns if c.startswith('is_')]
for col in pattern_cols:
    unique_vals = set(df[col].unique())
    is_binary = unique_vals.issubset({0, 1})
    hit_pct = df[col].mean() * 100
    status = "\u2705" if is_binary else "\u274c"
    print(f"  {status} {col}: {hit_pct:.2f}% of candles (binary={is_binary})")

# Anatomy stats
anatomy_cols = ['body_size', 'body_size_abs', 'upper_shadow', 'lower_shadow',
                'candle_range', 'body_to_range_ratio', 'shadow_ratio']
print(f"\nCandlestick Anatomy Stats:")
print(df[anatomy_cols].describe().round(4).to_string())


  VERIFICATION REPORT

Trend Distribution:
trend
Bullish    35580
Bearish    34428
Name: count, dtype: int64

NaN Check: 0 remaining NaNs

Candlestick Pattern Hit Rates:
  ✅ is_doji: 11.28% of candles (binary=True)
  ✅ is_hammer: 5.58% of candles (binary=True)
  ✅ is_inverted_hammer: 4.32% of candles (binary=True)
  ✅ is_engulfing_bullish: 8.92% of candles (binary=True)
  ✅ is_engulfing_bearish: 8.90% of candles (binary=True)
  ✅ is_morning_star: 2.06% of candles (binary=True)
  ✅ is_evening_star: 2.15% of candles (binary=True)
  ✅ is_marubozu: 0.80% of candles (binary=True)

Candlestick Anatomy Stats:
        body_size  body_size_abs  upper_shadow  lower_shadow  candle_range  body_to_range_ratio  shadow_ratio
count  70008.0000     70008.0000    70008.0000    70008.0000    70008.0000           70008.0000  7.000800e+04
mean       1.0552       145.1181       77.4270       85.9363      308.4814               0.4220  1.642583e+10
std      274.7157       233.2601      115.1577      149.113